# Showcase: our agents against the rule-based agent

Both final agents of the report play 1 vs. 1 against the rule-based agent, in the same setting as our
evaluation (17×17 board, crate density 0.35, 9 coins, 200 steps, always the best action):

- **A2C**: the FC128 (Kartal) network after the three curriculum stages on 17×17 (T03, section 6.3).
- **MCTS**: the same network guiding a search with 16 simulations per move (T04, section 6.4).

Each agent plays a few games, we print a summary, and then watch one game of each agent.
The weights come from `agent_code/`, where both agents are also packaged for the original game.

In [ ]:
import pathlib

import equinox as eqx
import jax
import numpy as np

from bomberman.agents import RuleBasedAgent
from bomberman.game import Game
from bomberman.trained import a2c_agent, load_network, mcts_agent
from bomberman.training import Rollout, versus_task_settings
from bomberman.training.scenarios import NoReward
from bomberman.visualize import animate

HERE = pathlib.Path.cwd().resolve()
ROOT = next(p for p in (HERE, *HERE.parents) if (p / "agent_code").is_dir())
print(f"backend: {jax.default_backend()}")

## Setting and agents

In [ ]:
SETTINGS = versus_task_settings(size=17, crate_density=0.35, coins=9, max_steps=200, agents=2)
ROLLOUT = Rollout(Game(settings=SETTINGS), NoReward(), SETTINGS.rules.max_steps)

AGENTS = {
    "A2C": a2c_agent(load_network(ROOT / "agent_code" / "a2c_kartal" / "weights.eqx", SETTINGS)),
    "MCTS": mcts_agent(load_network(ROOT / "agent_code" / "mcts_kartal" / "weights.eqx", SETTINGS), SETTINGS,
                       num_simulations=16),
}
OPPONENT = RuleBasedAgent()
GAMES = 16  # games per agent; MCTS is much slower, so use fewer on a CPU


@eqx.filter_jit
def play(agent, key):
    """Play `GAMES` games, our agent as player 0 and the rule-based agent as player 1."""
    return ROLLOUT([agent, OPPONENT], key, greedy=True, num_envs=GAMES)

## Games

The score is the game score (1 per coin, 5 per kill). An agent wins a game if its score is higher.

In [ ]:
rollouts = {}
for name, agent in AGENTS.items():
    out = rollouts[name] = play(agent, jax.random.key(0))
    score = np.asarray(out.states.score[:, -1])  # (games, seats)
    print(f"{name:5s} vs rule-based: score {score[:, 0].mean():.2f} vs {score[:, 1].mean():.2f} | "
          f"win {(score[:, 0] > score[:, 1]).mean():.2f} | coins {np.asarray(out.coins[:, 0]).mean():.2f} | "
          f"survived {np.asarray(out.alive[:, 0]).mean():.2f}")

## Replays

Agent 0 is our agent, agent 1 the rule-based agent. Use the slider to step through the game, and change `env`
to watch another game.

In [ ]:
animate(rollouts["A2C"], env=0)

In [ ]:
animate(rollouts["MCTS"], env=0)